# Phase 6 — Abstractive Legal Summarisation with the pre-trained **Encoder–Decoder** (Fusion-in-Decoder)

The Phase-1 T5-style model is already a seq2seq summariser: a bidirectional encoder reads the document and the decoder writes the summary through cross-attention. The only problem is length. The encoder was pre-trained on 512-token windows, and judgments are much longer, so it is used as **Fusion-in-Decoder (FiD)**:

```
document → K chunks (511 BPE tokens + </s>, spread uniformly over the judgment)
        → ENCODER on each chunk independently        (bidirectional, the 512 window it was pre-trained on)
        → concatenate all chunk states  [B, K·512, d] (+ key mask)
        → DECODER  <s> y_1 … y_L  cross-attends to ALL chunks at once → y_1 … y_L </s>
```

There are **no new weights and no re-pre-training**. Encoder cost grows linearly in K, and the decoder sees the whole selected document. Decoder length can exceed 512 because T5 relative-position buckets are log-spaced.

**Methods** (`method=`): `full` · `partial` · `lora` · `lorma` (LoRMA⁺, compulsory) · `qlora` (NF4 4-bit base + LoRA) · `houlsby` · `pfeiffer` · `prefix` (encoder + decoder self-attention) · `prompt` (soft tokens before every chunk) · `kd` (6-encoder/4-decoder student initialised from the fine-tuned teacher, CE + temperature-scaled KL).

**Engineering:** fused SDPA everywhere (dropout passed to the kernel), per-layer gradient checkpointing, and fp16/bf16 autocast. Incremental beam search uses a self-attention KV cache and computes cross-attention K/V **once per document**, shared by all beams. Beams are folded into the query axis, so the large cross cache is never copied.

**Run order:** top to bottom. Cell 2b self-tests the FiD runners against the original `LegalSeq2SeqLM.forward` and cached beam decoding against full decoding. Cell 5 checks that every PEFT module is the identity at initialisation. Then run the smoke test, then the experiments.

In [ ]:
# metrics used for local validation (the official scorer computes BERTScore + ROUGE the same way)
!pip -q install rouge-score bert-score

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
# ============================================================
# 0. SETUP
# ============================================================
import os, json, math, time, random, copy, pickle, glob, shutil, gc
from types import SimpleNamespace
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.checkpoint import checkpoint as grad_ckpt

from google.colab import drive
drive.mount('/content/drive')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "|", torch.cuda.get_device_name(0) if device.type == "cuda" else "")
AMP_DTYPE = (None if device.type != "cuda" else
             torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16)   # fp16 on T4
print("mixed precision:", AMP_DTYPE)

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = None
print("HF token available:", bool(HF_TOKEN))

# ---- paths (same layout as the Phase-5 encoder-decoder notebook; edit if yours differ) --------------
DRIVE_ROOT = "/content/drive/MyDrive/Phase1_Pretraining"
WORK_DIR   = os.path.join(DRIVE_ROOT, "phase1_encoder_decoder")
CKPT_DIR   = os.path.join(WORK_DIR, "checkpoints")
ckpt_path  = os.path.join(CKPT_DIR, "encoder_decoder_best.pt")
if not os.path.exists(ckpt_path):
    ckpt_path = os.path.join(CKPT_DIR, "encoder_decoder_latest.pt")
tok_path   = os.path.join(WORK_DIR, "final_bpe_en.json")
p6_dir     = os.path.join(WORK_DIR, "phase6"); os.makedirs(p6_dir, exist_ok=True)
local_ck   = "/content/p6_ckpt"; os.makedirs(local_ck, exist_ok=True)

PHASE1_NB = None
for pat in ("/content/drive/MyDrive/Colab Notebooks/phase1_encoder_decoder*.ipynb",
            os.path.join(WORK_DIR, "*.ipynb"), "/content/phase1_encoder_decoder*.ipynb"):
    hits = sorted(glob.glob(pat))
    if hits:
        PHASE1_NB = hits[0]; break
for what, p in (("checkpoint", ckpt_path), ("tokenizer", tok_path)):
    if not os.path.exists(p):
        raise FileNotFoundError(f"Phase-1 {what} not found: {p}  -> edit the paths above")
print("checkpoint:", ckpt_path, "| phase-1 notebook:", PHASE1_NB)


def load_definitions_from_notebook(nb_path):
    """exec the Phase-1 cells that define BPETokenizer and normalize_text (identical tokenisation)."""
    if nb_path is None:
        raise FileNotFoundError("Phase-1 notebook not found: set PHASE1_NB")
    nb = json.load(open(nb_path, encoding="utf-8"))
    picked = []
    for marker in ("class BPETokenizer", "def normalize_text"):
        for i, c in enumerate(nb["cells"]):
            if c["cell_type"] == "code" and marker in "".join(c["source"]) and i not in picked:
                picked.append(i); break
        else:
            raise RuntimeError(f"no cell containing {marker!r} in {nb_path}")
    for i in sorted(picked):
        src = "".join(nb["cells"][i]["source"])
        src = "\n".join(l for l in src.split("\n") if not l.lstrip().startswith(("!", "%")))
        exec(compile(src, f"<phase1 cell {i}>", "exec"), globals())
    print("loaded definitions from Phase-1 cells", sorted(picked))


if not all(n in globals() for n in ("BPETokenizer", "normalize_text")):
    load_definitions_from_notebook(PHASE1_NB)

tokenizer = BPETokenizer.from_json(tok_path)
PAD_ID, BOS_ID, EOS_ID = tokenizer.stoi["<pad>"], tokenizer.stoi["<s>"], tokenizer.stoi["</s>"]
print(f"tokenizer vocab: {len(tokenizer.itos):,} | PAD {PAD_ID}  BOS {BOS_ID}  EOS {EOS_ID}")

Mounted at /content/drive
device: cuda | Tesla T4
mixed precision: torch.float16
HF token available: True
checkpoint: /content/drive/MyDrive/Phase1_Pretraining/phase1_encoder_decoder/checkpoints/encoder_decoder_best.pt | phase-1 notebook: /content/drive/MyDrive/Colab Notebooks/phase1_encoder_decoder_bpe_pretraining_txl_data_pipeline-2.ipynb
loaded definitions from Phase-1 cells [7, 13]
tokenizer vocab: 32,768 | PAD 0  BOS 2  EOS 3


In [ ]:
# ============================================================
# 1. PRE-TRAINED ENCODER-DECODER  (architecture copied from Phase 1 so the state-dict loads strictly)
#
# Only two changes, both numerically equivalent to the original:
#   * Encoder.forward folds [relative-position bias + key-padding mask] into ONE additive bias, built once
#     per forward and shared by all layers, and attention uses F.scaled_dot_product_attention (fused,
#     no softmax matrix kept). FAST_ATTN["on"]=False restores the original manual attention.
#   * Encoder.forward(n_frozen=k): the first k layers run under no_grad (frozen part of a partly fine-tuned encoder).
# The decoder is untouched (we only ever run it for ONE query token, so it is cheap).
# ============================================================
FAST_ATTN = {"on": True}


class RMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d_model))
        self.eps = eps

    def forward(self, x):
        var = x.pow(2).mean(-1, keepdim=True)
        x = x * torch.rsqrt(var + self.eps)
        return self.weight * x


class RelativePositionBias(nn.Module):
    """T5-style relative position bias, bucketed on a log scale for far-apart positions."""

    def __init__(self, num_heads, num_buckets=32, max_distance=128, bidirectional=True):
        super().__init__()
        self.num_buckets = num_buckets
        self.max_distance = max_distance
        self.bidirectional = bidirectional
        self.embedding = nn.Embedding(num_buckets, num_heads)

    @staticmethod
    def _bucket(relative_position, bidirectional, num_buckets, max_distance):
        ret = 0
        n = -relative_position
        if bidirectional:
            num_buckets //= 2
            ret += (n < 0).to(torch.long) * num_buckets
            n = torch.abs(n)
        else:
            n = torch.clamp(n, min=0)
        max_exact = num_buckets // 2
        is_small = n < max_exact
        val_if_large = max_exact + (
            torch.log(n.float() / max_exact + 1e-6)
            / math.log(max_distance / max_exact)
            * (num_buckets - max_exact)
        ).to(torch.long)
        val_if_large = torch.clamp(val_if_large, max=num_buckets - 1)
        return ret + torch.where(is_small, n, val_if_large)

    def forward(self, q_len, k_len, device):
        q_pos = torch.arange(q_len, device=device)[:, None]
        k_pos = torch.arange(k_len, device=device)[None, :]
        rel_pos = k_pos - q_pos
        bucket = self._bucket(rel_pos, self.bidirectional, self.num_buckets, self.max_distance)
        values = self.embedding(bucket)                    # q_len, k_len, num_heads
        return values.permute(2, 0, 1).unsqueeze(0)        # 1, num_heads, q_len, k_len


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.d_head = d_model // num_heads
        self.q = nn.Linear(d_model, d_model, bias=False)
        self.k = nn.Linear(d_model, d_model, bias=False)
        self.v = nn.Linear(d_model, d_model, bias=False)
        self.o = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x_q, x_kv, mask=None, position_bias=None, attn_bias=None):
        B, Lq, D = x_q.shape
        Lk = x_kv.shape[1]
        q = self.q(x_q).view(B, Lq, self.num_heads, self.d_head).transpose(1, 2)
        k = self.k(x_kv).view(B, Lk, self.num_heads, self.d_head).transpose(1, 2)
        v = self.v(x_kv).view(B, Lk, self.num_heads, self.d_head).transpose(1, 2)

        if attn_bias is not None:            # fused path: softmax(q k^T / sqrt(d_head) + attn_bias) v
            out = F.scaled_dot_product_attention(
                q, k, v, attn_mask=attn_bias.to(q.dtype),
                dropout_p=self.dropout.p if self.training else 0.0)
        else:                                # original path
            scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_head)
            if position_bias is not None:
                scores = scores + position_bias
            if mask is not None:
                scores = scores.masked_fill(mask == 0, float("-inf"))
            attn = self.dropout(torch.softmax(scores, dim=-1))
            out = torch.matmul(attn, v)
        return self.o(out.transpose(1, 2).contiguous().view(B, Lq, D))


class GatedFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.wi_0 = nn.Linear(d_model, d_ff, bias=False)
        self.wi_1 = nn.Linear(d_model, d_ff, bias=False)
        self.wo = nn.Linear(d_ff, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.act = nn.GELU()

    def forward(self, x):
        h = self.act(self.wi_0(x)) * self.wi_1(x)
        return self.wo(self.dropout(h))


class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.ln1 = RMSNorm(d_model)
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ln2 = RMSNorm(d_model)
        self.ff = GatedFeedForward(d_model, d_ff, dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask, position_bias, attn_bias=None):
        h = self.ln1(x)
        x = x + self.dropout(self.self_attn(h, h, mask, position_bias, attn_bias))
        h = self.ln2(x)
        x = x + self.dropout(self.ff(h))
        return x


class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.ln1 = RMSNorm(d_model)
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ln2 = RMSNorm(d_model)
        self.cross_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ln3 = RMSNorm(d_model)
        self.ff = GatedFeedForward(d_model, d_ff, dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_out, self_mask, cross_mask, self_pos_bias):
        h = self.ln1(x)
        x = x + self.dropout(self.self_attn(h, h, self_mask, self_pos_bias))
        h = self.ln2(x)
        x = x + self.dropout(self.cross_attn(h, enc_out, cross_mask, None))
        h = self.ln3(x)
        x = x + self.dropout(self.ff(h))
        return x


class Encoder(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, num_layers, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList(
            [EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)]
        )
        self.pos_bias = RelativePositionBias(num_heads, bidirectional=True)
        self.final_ln = RMSNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, attention_mask, n_frozen=0):
        L = x.shape[1]
        pos_bias = self.pos_bias(L, L, x.device)                      # [1,H,L,L]
        mask = attention_mask[:, None, None, :]                       # [B,1,1,L]
        attn_bias = None
        if FAST_ATTN["on"]:
            with torch.no_grad():                                     # dtype the attention will run in (fp32 or autocast)
                dt = self.layers[0].self_attn.q(x[:, :1]).dtype
            keep = attention_mask.bool()[:, None, None, :]
            attn_bias = pos_bias.to(dt).expand(x.size(0), -1, -1, -1).masked_fill(~keep, float("-inf"))   # [B,H,L,L], built once, shared by all layers
        x = self.dropout(x)
        for i, layer in enumerate(self.layers):
            if i < n_frozen:                                          # frozen bottom layers: no graph, no memory
                with torch.no_grad():
                    x = layer(x, mask, pos_bias, attn_bias)
            else:
                x = layer(x, mask, pos_bias, attn_bias)
        return self.final_ln(x)


class Decoder(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, num_layers, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList(
            [DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)]
        )
        self.pos_bias = RelativePositionBias(num_heads, bidirectional=False)
        self.final_ln = RMSNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_out, self_attention_mask, cross_attention_mask):
        L = x.shape[1]
        causal = torch.tril(torch.ones(L, L, device=x.device, dtype=torch.bool))
        pad = self_attention_mask[:, None, None, :].bool()
        self_mask = (causal[None, None] & pad).float()
        cross_mask = cross_attention_mask[:, None, None, :]
        pos_bias = self.pos_bias(L, L, x.device)
        x = self.dropout(x)
        for layer in self.layers:
            x = layer(x, enc_out, self_mask, cross_mask, pos_bias)
        return self.final_ln(x)


class LegalSeq2SeqLM(nn.Module):
    def __init__(self, vocab_size, d_model=512, num_heads=16, d_ff=2048,
                 num_encoder_layers=9, num_decoder_layers=9, dropout=0.1, pad_id=0):
        super().__init__()
        self.shared_embed = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.encoder = Encoder(d_model, num_heads, d_ff, num_encoder_layers, dropout)
        self.decoder = Decoder(d_model, num_heads, d_ff, num_decoder_layers, dropout)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.shared_embed.weight  # tied embeddings

    def forward(self, input_ids, attention_mask, decoder_input_ids, decoder_attention_mask):
        enc_out = self.encoder(self.shared_embed(input_ids), attention_mask)
        dec_out = self.decoder(
            self.shared_embed(decoder_input_ids), enc_out, decoder_attention_mask, attention_mask
        )
        return self.lm_head(dec_out)


def load_pretrained_ed():
    ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    c = ck.get("model_config", {})
    m = LegalSeq2SeqLM(
        vocab_size=c.get("vocab_size", len(tokenizer.itos)), d_model=c.get("d_model", 512),
        num_heads=c.get("num_heads", 16), d_ff=c.get("d_ff", 2048),
        num_encoder_layers=c.get("num_encoder_layers", 9), num_decoder_layers=c.get("num_decoder_layers", 9),
        dropout=c.get("dropout", 0.1), pad_id=c.get("pad_id", PAD_ID))
    if m.shared_embed.num_embeddings != len(tokenizer.itos):
        raise ValueError(f"model vocab {m.shared_embed.num_embeddings} != tokenizer vocab {len(tokenizer.itos)}: "
                         f"wrong tokenizer file for this checkpoint")
    m.load_state_dict(ck["model_state_dict"])            # strict: fails loudly on any mismatch
    print(f"loaded Phase-1 checkpoint | step {ck.get('global_step')} | best val token-loss {ck.get('best_val_loss')}")
    return m.eval()


ED = load_pretrained_ed().to(device)
D_ED = ED.shared_embed.embedding_dim
print(f"Encoder-decoder loaded: {sum(p.numel() for p in ED.parameters())/1e6:.1f}M params | d_model {D_ED} | "
      f"{len(ED.encoder.layers)} enc + {len(ED.decoder.layers)} dec layers")

loaded Phase-1 checkpoint | step 11200 | best val token-loss 4.871949663513102
Encoder-decoder loaded: 101.7M params | d_model 512 | 9 enc + 9 dec layers


In [ ]:
# ============================================================
# DATA: Phase-6 (abstractive summarisation) from the course Hugging Face repo
# ============================================================
from huggingface_hub import hf_hub_download, HfApi

REPO_ID = "Exploration-Lab/CS-781-Capstone"


def find_repo_file(basename):
    """Locate e.g. train.json inside the Phase-6 folder without hard-coding its exact name."""
    files = HfApi().list_repo_files(REPO_ID, repo_type="dataset", token=HF_TOKEN)
    hits = [f for f in files if f.endswith("/" + basename) and ("summ" in f.lower() or "phase_6" in f.lower())]
    if len(hits) != 1:
        raise RuntimeError(f"expected exactly one {basename} for Phase 6, got {hits}. "
                           f"Set the path by hand. Phase-6-like files: {[f for f in files if 'summ' in f.lower()]}")
    return hits[0]


_DOC_KEYS = ("document", "text", "judgment", "judgement", "doc", "source")
_SUM_KEYS = ("summary", "abstract", "target", "reference")


def _pick(r, keys, what, rid):
    for k in keys:
        if k in r and r[k] is not None:
            v = r[k]
            return " ".join(v) if isinstance(v, list) else str(v)
    if what == "summary":
        return None
    raise KeyError(f"record {rid}: no {what} field among {keys}; keys present: {list(r)}")


def read_split(filename):
    path = hf_hub_download(REPO_ID, find_repo_file(filename), repo_type="dataset", token=HF_TOKEN)
    raw = json.load(open(path, encoding="utf-8"))
    if isinstance(raw, dict):                          # {id: {...}} or {id: "text"}
        raw = [({**v, "id": k} if isinstance(v, dict) else {"id": k, "document": v}) for k, v in raw.items()]
    recs = []
    for r in raw:
        rid = str(r["id"])
        recs.append(dict(id=rid, doc=_pick(r, _DOC_KEYS, "document", rid), summ=_pick(r, _SUM_KEYS, "summary", rid)))
    return recs


# all_train_recs = read_split("train.json")

_DOC_KEYS = ("document", "text", "judgment", "judgement", "doc", "source", "input", "article", "case_text")
_SUM_KEYS = ("summary", "summaries", "abstract", "target", "reference", "headnote", "output", "gold", "label")


def _as_text(v):
    if v is None:
        return None
    if isinstance(v, list):
        v = " ".join(str(x) for x in v)
    v = str(v).strip()
    return v or None


def read_split(filename):
    path = hf_hub_download(REPO_ID, find_repo_file(filename), repo_type="dataset", token=HF_TOKEN)
    raw = json.load(open(path, encoding="utf-8"))
    if isinstance(raw, dict):
        raw = [({**v, "id": k} if isinstance(v, dict) else {"id": k, "document": v}) for k, v in raw.items()]
    keys = list(raw[0].keys())
    print(f"{filename}: {len(raw)} records | fields: {keys}")
    # document = first known doc key present, else the longest text field; summary = first known summary key present
    doc_k = next((k for k in _DOC_KEYS if k in keys), None)
    sum_k = next((k for k in _SUM_KEYS if k in keys), None)
    if doc_k is None:
        text_keys = [k for k in keys if k != "id" and isinstance(raw[0][k], (str, list))]
        doc_k = max(text_keys, key=lambda k: len(_as_text(raw[0][k]) or ""))
    print(f"  using document field '{doc_k}', summary field '{sum_k}'")
    return [dict(id=str(r.get("id", i)), doc=_as_text(r.get(doc_k)) or "",
                 summ=_as_text(r.get(sum_k)) if sum_k else None) for i, r in enumerate(raw)]


all_train_recs = read_split("train.json")
test_recs = read_split("test.json")
_n_missing = sum(r["summ"] is None for r in all_train_recs)
if _n_missing == len(all_train_recs):
    raise KeyError("no summary field found in train.json: look at the 'fields' printed above and add the "
                   "summary field's name to _SUM_KEYS")
if _n_missing:
    print(f"dropping {_n_missing} training records with an empty summary")
all_train_recs = [r for r in all_train_recs if r["summ"] is not None and r["doc"]]

print(f"{len(all_train_recs)} train documents | {len(test_recs)} test documents")

# ---- no official dev split for Phase 6: hold out a validation split from train ----------------
VAL_FRAC, SPLIT_SEED = 0.05, 1234
_perm = np.random.RandomState(SPLIT_SEED).permutation(len(all_train_recs))
_n_val = max(50, int(round(VAL_FRAC * len(all_train_recs))))
val_recs = [all_train_recs[i] for i in _perm[:_n_val]]
train_recs = [all_train_recs[i] for i in _perm[_n_val:]]
print(f"split -> train {len(train_recs)} | val {len(val_recs)}  (val is held out from train.json)")

# ---- tokenise once, cache to Drive (no special tokens: added per chunk / per sequence later) ----
# NORMALIZE_TARGETS: apply the Phase-1 normalize_text to the reference summaries too. Keep True if
# normalize_text only cleans whitespace/unicode; set False if it lower-cases (the scorer compares
# against the raw references, so a lower-cased output would cost BERTScore).
NORMALIZE_TARGETS = True


def _tok_cached(texts, tag, norm=True, bs=1000, workers=None):
    workers = workers or min(4, os.cpu_count() or 1)
    path = os.path.join(p6_dir, f"tok_{tag}.pkl")
    if os.path.exists(path):
        return pickle.load(open(path, "rb"))
    out = []
    for i in range(0, len(texts), bs):
        part = [normalize_text(t) if norm else t for t in texts[i:i + bs]]
        out += [np.asarray(x, dtype=np.int32) for x in
                tokenizer.encode_batch(part, add_special_tokens=False, num_workers=workers, chunk_size=16)]
        print(f"  tokenised {min(i + bs, len(texts)):,}/{len(texts):,} ({tag})")
    pickle.dump(out, open(path, "wb"))
    return out


def _tok_split(recs, tag):
    docs = _tok_cached([r["doc"] for r in recs], f"{tag}_doc_s{SPLIT_SEED}")
    sums = None
    if recs[0]["summ"] is not None:
        sums = _tok_cached([r["summ"] for r in recs], f"{tag}_sum_s{SPLIT_SEED}_n{int(NORMALIZE_TARGETS)}",
                           norm=NORMALIZE_TARGETS)
    return docs, sums


tr_doc, tr_sum = _tok_split(train_recs, "train")
va_doc, va_sum = _tok_split(val_recs, "val")
te_doc, _ = _tok_split(test_recs, "test")

# ---- length statistics: these decide the generation lengths and the context layout ------------
dl = np.array([len(x) for x in tr_doc]); sl = np.array([len(x) for x in tr_sum])
sw = np.array([len(r["summ"].split()) for r in train_recs])
pct = lambda a: "  ".join(f"p{p} {int(np.percentile(a, p))}" for p in (10, 50, 90, 95, 99))
print("document BPE tokens :", pct(dl), f" max {dl.max()}")
print("summary  BPE tokens :", pct(sl), f" max {sl.max()}")
print("summary  words      :", pct(sw))
print(f"tokens per summary word ~ {sl.sum() / max(sw.sum(), 1):.2f} | summary/document length ratio median "
      f"{np.median(sl / np.maximum(dl, 1)):.3f}")
print(f"512-token chunks per document: median {np.median(np.ceil(dl / 511)):.0f}  p90 {np.percentile(np.ceil(dl / 511), 90):.0f}")

SUM_TOK_P10, SUM_TOK_P50, SUM_TOK_P90, SUM_TOK_P95 = (int(np.percentile(sl, p)) for p in (10, 50, 90, 95))
SUM_WORDS_P50 = int(np.median(sw))

Phase_6_Summ/Train/train.json: reconstructing file:   0%|          |  0.00B /  211MB            

Phase_6_Summ/Train/train.json: downloading bytes:           |  0.00B            

train.json: 7030 records | fields: ['id', 'document', 'summary']
  using document field 'document', summary field 'summary'


test.json:   0%|          | 0.00/2.73M [00:00<?, ?B/s]

test.json: 100 records | fields: ['id', 'document']
  using document field 'document', summary field 'None'
dropping 2 training records with an empty summary
7028 train documents | 100 test documents
split -> train 6677 | val 351  (val is held out from train.json)
document BPE tokens : p10 1572  p50 3652  p90 9088  p95 13032  p99 28019  max 162005
summary  BPE tokens : p10 340  p50 749  p90 1763  p95 2390  p99 5088  max 32291
summary  words      : p10 292  p50 638  p90 1493  p95 2007  p99 4267
tokens per summary word ~ 1.18 | summary/document length ratio median 0.219
512-token chunks per document: median 8  p90 18


In [ ]:
# ============================================================
# Model-agnostic utilities: chunking, detokenisation, post-processing, ROUGE / BERTScore, submission
# ============================================================
import re, zipfile, collections


# ---------------- long-document chunking (as in Phase 5) ----------------
def chunk_ids(ids, body_len=511, stride=511, min_tail=16):
    if len(ids) <= body_len:
        return [ids]
    chunks, s = [], 0
    while True:
        chunks.append(ids[s:s + body_len])
        if s + body_len >= len(ids):
            break
        s += stride
    if len(chunks) > 1 and len(chunks[-1]) < min_tail:
        chunks.pop()
    return chunks


def select_chunks(chunks, max_chunks, strategy="uniform"):
    """uniform: spread the budget over the whole judgment (facts at the start, reasoning in the middle,
    the order at the end all matter for a summary). head / tail / head_tail kept for the ablation."""
    n = len(chunks)
    if n <= max_chunks:
        return chunks
    if strategy == "head":
        return chunks[:max_chunks]
    if strategy == "tail":
        return chunks[-max_chunks:]
    if strategy == "head_tail":
        t = max_chunks // 2
        return chunks[:max_chunks - t] + (chunks[-t:] if t else [])
    if strategy == "uniform":
        idx = np.unique(np.linspace(0, n - 1, max_chunks).round().astype(int))
        return [chunks[i] for i in idx]
    raise ValueError(strategy)


# ---------------- detokenisation + post-processing ----------------
_FIXES = [(re.compile(r"\s+([.,;:!?%)\]}'])"), r"\1"),      # 'court .'  -> 'court.'
          (re.compile(r"([(\[{])\s+"), r"\1"),               # '( a )'    -> '(a)'
          (re.compile(r"(?<=\w)\s*([-/])\s*(?=\w)"), r"\1"),  # '302 / 34' -> '302/34', 'Section - 3' -> 'Section-3'
          (re.compile(r"\s{2,}"), " ")]


def detok(ids):
    """BPE ids -> text. tokenizer.decode() puts a space after every word-final token including punctuation
    ('court . the'), so spacing is repaired here. Special tokens are dropped."""
    ids = [int(i) for i in ids if int(i) not in (PAD_ID, BOS_ID, EOS_ID)]
    s = tokenizer.decode(ids)
    for pat, rep in _FIXES:
        s = pat.sub(rep, s)
    return s.strip()


def postprocess(text, max_words=None):
    """drop exact/near-exact repeated sentences (small models loop), cap length, end on a sentence boundary"""
    sents = re.split(r"(?<=[.!?])\s+", text.strip())
    seen, out = set(), []
    for s in sents:
        key = re.sub(r"\W+", " ", s.lower()).strip()
        if len(key) < 2 or key in seen:
            continue
        seen.add(key); out.append(s)
    text = " ".join(out)
    if max_words:
        w = text.split()
        if len(w) > max_words:
            text = " ".join(w[:max_words])
    cut = max(text.rfind(". "), text.rfind("."))
    if cut > 0.8 * len(text) and cut < len(text) - 1:     # trailing fragment after the last full stop
        text = text[:cut + 1]
    return text.strip() or "."


# ---------------- metrics ----------------
try:
    from rouge_score import rouge_scorer as _rs
    _ROUGE = _rs.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
except Exception:
    _ROUGE = None


def _toks(s):
    return re.findall(r"[a-z0-9]+", s.lower())


def _f1(p, r):
    return 0.0 if p + r == 0 else 2 * p * r / (p + r)


def _rouge_n(a, b, n):
    ca = collections.Counter(zip(*[a[i:] for i in range(n)])); cb = collections.Counter(zip(*[b[i:] for i in range(n)]))
    ov = sum((ca & cb).values())
    return _f1(ov / max(sum(ca.values()), 1), ov / max(sum(cb.values()), 1))


def _rouge_l(a, b):
    if not a or not b:
        return 0.0
    prev = [0] * (len(b) + 1)
    for x in a:
        cur = [0]
        for j, y in enumerate(b):
            cur.append(prev[j] + 1 if x == y else max(prev[j + 1], cur[j]))
        prev = cur
    l = prev[-1]
    return _f1(l / len(a), l / len(b))


def rouge_scores(preds, refs):
    r1 = r2 = rl = 0.0
    for p, r in zip(preds, refs):
        if _ROUGE is not None:
            s = _ROUGE.score(r, p)
            r1 += s["rouge1"].fmeasure; r2 += s["rouge2"].fmeasure; rl += s["rougeL"].fmeasure
        else:                                            # fallback (no stemming): numbers differ slightly
            a, b = _toks(p), _toks(r)
            r1 += _rouge_n(a, b, 1); r2 += _rouge_n(a, b, 2); rl += _rouge_l(a, b)
    n = max(len(preds), 1)
    return dict(rouge1=r1 / n, rouge2=r2 / n, rougeL=rl / n)


BERTSCORE_KW = dict(lang="en")      # the course scorer's settings are not published; lang="en" = roberta-large
_BS = {}


def bertscore_f1(preds, refs, batch_size=16):
    """Evaluation only (the official scorer does the same). Returns None if bert-score is unavailable."""
    try:
        from bert_score import BERTScorer
    except Exception:
        return None
    key = json.dumps(BERTSCORE_KW, sort_keys=True)
    if key not in _BS:
        _BS[key] = BERTScorer(device=str(device), **BERTSCORE_KW)
    _, _, f = _BS[key].score(preds, refs, batch_size=batch_size)
    return float(f.mean())


def evaluate_texts(preds, refs, bert=True):
    m = rouge_scores(preds, refs)
    m["bertscore"] = bertscore_f1(preds, refs) if bert else None
    m["avg_words"] = float(np.mean([len(p.split()) for p in preds]))
    m["ref_words"] = float(np.mean([len(r.split()) for r in refs]))
    return m


def fmt_metrics(m):
    bs = "n/a" if m.get("bertscore") is None else f"{m['bertscore']:.4f}"
    return (f"BERTScore {bs} | R1 {m['rouge1']:.4f} R2 {m['rouge2']:.4f} RL {m['rougeL']:.4f} | "
            f"words {m['avg_words']:.0f} (ref {m['ref_words']:.0f})")


# ---------------- submission ----------------
def write_submission(preds_by_id, name):
    out = [dict(id=r["id"], summary=preds_by_id[r["id"]]) for r in test_recs]
    assert len(out) == len(test_recs) == len({o["id"] for o in out}), "need exactly one summary per test id"
    assert all(isinstance(o["summary"], str) and o["summary"].strip() for o in out), "empty summary"
    d = os.path.join(p6_dir, f"submission_{name}"); os.makedirs(d, exist_ok=True)
    jp = os.path.join(d, "prediction.json")
    json.dump(out, open(jp, "w", encoding="utf-8"), ensure_ascii=False)
    zp = os.path.join(d, "submission.zip")
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(jp, arcname="prediction.json")          # at the ROOT of the zip, as Codabench requires
    print(f"wrote {len(out)} summaries (avg {np.mean([len(o['summary'].split()) for o in out]):.0f} words) -> {zp}")
    return zp

In [ ]:
# ============================================================
# Generic batched beam search (model-agnostic)
#
# The model side supplies three closures:
#   start()            -> (state, logits[B,V])   consume the prompt / encode the source, logits for the 1st token
#   step(state, tok)   -> (state, logits[N,V])   consume one token per hypothesis (N = B*beams)
#   reorder(state, ix) -> state                  keep hypothesis rows ix (rows never move to another source:
#                                                row r always belongs to source r // beams)
# Features: length penalty, min length, no-repeat-ngram blocking, repetition penalty, early stopping.
# num_beams=1 gives greedy decoding.
# ============================================================
@torch.no_grad()
def beam_search(start, step, reorder, num_beams=4, max_new=256, min_new=0, len_penalty=1.0,
                no_repeat_ngram=3, rep_penalty=1.0, eos_id=None, banned=()):
    eos_id = EOS_ID if eos_id is None else eos_id
    state, logits = start()
    B, V = logits.shape
    K, dev = num_beams, logits.device
    state = reorder(state, torch.arange(B, device=dev).repeat_interleave(K))
    logits = logits.repeat_interleave(K, 0)
    beam_scores = torch.zeros(B, K, device=dev)
    beam_scores[:, 1:] = -1e9                       # identical beams at t=0: keep only the first
    seqs = [[] for _ in range(B * K)]
    finished = [[] for _ in range(B)]               # (normalised score, tokens)
    done = [False] * B
    banned = [t for t in banned if t is not None and t != eos_id]

    for t in range(max_new):
        lp = torch.log_softmax(logits.float(), -1)
        if banned:
            lp[:, banned] = -float("inf")
        if t < min_new:
            lp[:, eos_id] = -float("inf")
        if rep_penalty != 1.0 and t > 0:
            for r, s in enumerate(seqs):
                if s:
                    ix = torch.tensor(sorted(set(s)), device=dev)
                    lp[r, ix] -= math.log(rep_penalty)
        n = no_repeat_ngram
        if n and t >= n - 1:
            for r, s in enumerate(seqs):
                if done[r // K]:
                    continue
                prefix = tuple(s[len(s) - n + 1:]) if n > 1 else ()
                bad = [s[i + n - 1] for i in range(len(s) - n + 1) if tuple(s[i:i + n - 1]) == prefix]
                if bad:
                    lp[r, bad] = -float("inf")

        cand = (beam_scores.view(-1, 1) + lp).view(B, K * V)
        top_s, top_i = cand.topk(2 * K, dim=1)
        top_s, top_i = top_s.tolist(), top_i.tolist()
        sel_rows, sel_tok, new_scores = [], [], []
        for b in range(B):
            chosen = []
            if not done[b]:
                for sc, ix in zip(top_s[b], top_i[b]):
                    beam, tok = divmod(ix, V)
                    if sc < -1e8:
                        break
                    if tok == eos_id:
                        finished[b].append((sc / ((t + 1) ** len_penalty), list(seqs[b * K + beam])))
                    else:
                        chosen.append((b * K + beam, tok, sc))
                    if len(chosen) == K:
                        break
                # early stop: K finished hypotheses and no live beam can beat the worst of them
                if len(finished[b]) >= K:
                    best_live = max((c[2] for c in chosen), default=-1e9) / ((t + 1) ** len_penalty)
                    if best_live <= sorted(f[0] for f in finished[b])[-K]:
                        done[b] = True
            while len(chosen) < K:                    # finished source or not enough candidates: dummy rows
                chosen.append((b * K, eos_id if done[b] else (chosen[0][1] if chosen else eos_id), -1e9))
            for row, tok, sc in chosen:
                sel_rows.append(row); sel_tok.append(tok); new_scores.append(sc)
        if all(done):
            break
        seqs = [seqs[r] + [tk] for r, tk in zip(sel_rows, sel_tok)]
        beam_scores = torch.tensor(new_scores, device=dev).view(B, K)
        rows = torch.tensor(sel_rows, device=dev)
        state = reorder(state, rows)
        state, logits = step(state, torch.tensor(sel_tok, device=dev))

    out = []
    L = len(seqs[0]) if seqs else 0
    for b in range(B):
        if not done[b]:
            for k in range(K):
                sc = float(beam_scores[b, k])
                if sc > -1e8:
                    finished[b].append((sc / (max(L, 1) ** len_penalty), seqs[b * K + k]))
        out.append(max(finished[b], key=lambda x: x[0])[1] if finished[b] else [])
    return out

In [ ]:
# ============================================================
# Parameter-efficient adaptation modules (own implementations, no peft library)
#   LoRA     (Hu et al., 2021)            W x + (a/r) B A x                      B = 0 at init
#   LoRMA+   (Bihany, Patel, Modi, 2025)   (I + (a/r) B A) W0 x                   multiplicative, additive rank
#                                          inflation; B = 0 at init -> identity; computed as y0 + s*B(A y0),
#                                          y0 = W0 x, so no d x d matrix is ever formed (the paper's re-ordering)
#   QLoRA    (Dettmers et al., 2023)       frozen base stored as 4-bit NF4 (blockwise absmax), LoRA on top
#   Adapters (Houlsby et al., 2019 / Pfeiffer et al., 2021)  bottleneck MLP, zero-init up-projection
#   Prefix   (Li & Liang, 2021)            learned per-layer key/value prefixes (built in the model code)
#   Prompt   (Lester et al., 2021)         learned soft input tokens (built in the model code)
# ============================================================
class LoRALinear(nn.Module):
    def __init__(self, base, r=8, alpha=16, dropout=0.05):
        super().__init__()
        self.base = base
        for p in base.parameters():
            p.requires_grad = False
        self.in_features, self.out_features = base.in_features, base.out_features
        self.A = nn.Parameter(torch.empty(r, self.in_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(self.out_features, r))
        self.scale, self.drop = alpha / r, nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + self.scale * F.linear(F.linear(self.drop(x), self.A.to(x.dtype)), self.B.to(x.dtype))


class LoRMALinear(nn.Module):
    """LoRMA+ : W = (I_out + (a/r) B A) W0 ;  B in R^{out x r}, A in R^{r x out}. The bias is added after the
    multiplicative transform (it is not part of W0)."""

    def __init__(self, base, r=8, alpha=16, dropout=0.05):
        super().__init__()
        self.base = base
        for p in base.parameters():
            p.requires_grad = False
        self.in_features, self.out_features = base.in_features, base.out_features
        self.A = nn.Parameter(torch.empty(r, self.out_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(self.out_features, r))
        self.scale, self.drop = alpha / r, nn.Dropout(dropout)

    def forward(self, x):
        y0 = F.linear(x, self.base.weight.to(x.dtype))
        y = y0 + self.scale * F.linear(F.linear(self.drop(y0), self.A.to(y0.dtype)), self.B.to(y0.dtype))
        return y if self.base.bias is None else y + self.base.bias.to(y.dtype)


NF4_CODE = torch.tensor([-1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453,
                         -0.28444138169288635, -0.18477343022823334, -0.09105003625154495, 0.0,
                         0.07958029955625534, 0.16093020141124725, 0.24611230194568634, 0.33791524171829224,
                         0.44070982933044434, 0.5626170039176941, 0.7229568362236023, 1.0])


class NF4Linear(nn.Module):
    """Frozen linear layer stored as 4-bit NormalFloat codes (2 per byte) + one fp16 absmax per block of 64.
    Dequantised on the fly in forward (memory saving of QLoRA, without bitsandbytes)."""

    def __init__(self, base, block=64):
        super().__init__()
        W = base.weight.detach().float()
        self.shape, self.block = tuple(W.shape), block
        flat = W.reshape(-1)
        pad = (-flat.numel()) % block
        flat = torch.cat([flat, flat.new_zeros(pad)]).view(-1, block)
        absmax = flat.abs().amax(1, keepdim=True).clamp_min(1e-8)
        code = NF4_CODE.to(flat.device)
        idx = torch.empty(flat.shape, dtype=torch.uint8, device=flat.device)
        for i in range(0, flat.size(0), 4096):                       # chunked argmin keeps memory small
            idx[i:i + 4096] = ((flat[i:i + 4096] / absmax[i:i + 4096]).unsqueeze(-1) - code).abs().argmin(-1)
        idx = idx.view(-1)
        if idx.numel() % 2:
            idx = torch.cat([idx, idx.new_zeros(1)])
        self.register_buffer("packed", (idx[0::2] << 4) | idx[1::2])
        self.register_buffer("absmax", absmax.half())
        self.register_buffer("code", code.clone())
        self.n = W.numel()
        self.bias = None if base.bias is None else nn.Parameter(base.bias.detach().clone(), requires_grad=False)
        self.in_features, self.out_features = self.shape[1], self.shape[0]

    def dequant(self, dtype):
        idx = torch.stack([self.packed >> 4, self.packed & 15], 1).view(-1).long()
        w = self.code[idx][: self.absmax.numel() * self.block].view(-1, self.block) * self.absmax.float()
        return w.view(-1)[: self.n].view(self.shape).to(dtype)

    def forward(self, x):
        return F.linear(x, self.dequant(x.dtype), None if self.bias is None else self.bias.to(x.dtype))


class Adapter(nn.Module):
    """bottleneck adapter: x + up(act(down(x))); up is zero-initialised so the model starts unchanged"""

    def __init__(self, d, bottleneck=64, dropout=0.1):
        super().__init__()
        self.down, self.up = nn.Linear(d, bottleneck), nn.Linear(bottleneck, d)
        self.drop = nn.Dropout(dropout)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)

    def forward(self, x):
        return x + self.drop(self.up(F.gelu(self.down(x))))


class AdapterAfter(nn.Module):
    """Applies an adapter to the output of a sub-layer (Houlsby: after attention and FFN; Pfeiffer: after FFN).
    Works for sub-layers that return a tensor or a tuple (tensor, cache)."""

    def __init__(self, inner, d, bottleneck=64, dropout=0.1):
        super().__init__()
        self.inner, self.adapter = inner, Adapter(d, bottleneck, dropout)

    def forward(self, *a, **k):
        out = self.inner(*a, **k)
        if isinstance(out, tuple):
            return (self.adapter(out[0]),) + tuple(out[1:])
        return self.adapter(out)


def replace_linears(root, make, keep):
    """Replace every nn.Linear child c (full name n) under root with make(c) when keep(n) is True."""
    done = []
    for name, mod in list(root.named_modules()):
        for cname, child in list(mod.named_children()):
            full = f"{name}.{cname}" if name else cname
            if type(child) is nn.Linear and keep(full):
                setattr(mod, cname, make(child)); done.append(full)
    return done


def n_params(m, trainable=True):
    return sum(p.numel() for p in m.parameters() if (p.requires_grad or not trainable))


def freeze(m):
    for p in m.parameters():
        p.requires_grad = False

In [ ]:
# ============================================================
# Training / evaluation loop (shared by every method; the model file supplies the Summarizer)
#
# A Summarizer exposes:
#   sm.loss(batch, teacher=None)      -> scalar loss (teacher != None -> knowledge distillation)
#   sm.generate(batch, gen)           -> list of generated token-id lists
#   sm.pretrained_param_ids           -> ids of parameters that came from the Phase-1 checkpoint
# and the notebook supplies make_loader(split, cfg, shuffle, gen=False).
# ============================================================
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


def amp_ctx(on=True):
    return torch.autocast(device.type, dtype=AMP_DTYPE or torch.bfloat16, enabled=bool(on and AMP_DTYPE is not None))


def trainable_state(sm):
    return {n: p.detach().cpu().clone() for n, p in sm.named_parameters() if p.requires_grad}


def save_trainable(sm, path, extra=None):
    torch.save({"trainable": trainable_state(sm), **(extra or {})}, path)


def load_trainable(sm, path):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    missing = [n for n, p in sm.named_parameters() if p.requires_grad and n not in ck["trainable"]]
    if missing:
        raise RuntimeError(f"checkpoint {path} lacks {len(missing)} trainable tensors, e.g. {missing[:3]} "
                           f"(was it saved with the same method/config?)")
    sm.load_state_dict(ck["trainable"], strict=False)
    return ck


@torch.no_grad()
def eval_loss(sm, loader, max_batches=None):
    sm.eval(); tot, n = 0.0, 0
    for i, b in enumerate(loader):
        if max_batches and i >= max_batches:
            break
        with amp_ctx():
            tot += float(sm.loss(b)); n += 1
    return tot / max(n, 1)


@torch.no_grad()
def generate_split(sm, split, cfg, n_docs=None, gen=None):
    """-> {id: summary text}"""
    sm.eval()
    gen = {**GEN, **(gen or {})}
    loader = make_loader(split, cfg, shuffle=False, gen=True, limit=n_docs)
    out, t0 = {}, time.time()
    for b in loader:
        with amp_ctx():
            seqs = sm.generate(b, gen)
        for rid, s in zip(b["ids"], seqs):
            out[rid] = postprocess(detok(s), gen.get("max_words"))
        if len(out) % (len(b["ids"]) * 10) == 0:
            print(f"    generated {len(out)} ({time.time() - t0:.0f}s)")
    return out


def score_split(preds, recs, bert=True):
    refs = {r["id"]: r["summ"] for r in recs}
    ids = [i for i in preds if i in refs]
    return evaluate_texts([preds[i] for i in ids], [refs[i] for i in ids], bert=bert)


def train_summarizer(sm, cfg, tag, teacher=None):
    tr, va = make_loader("train", cfg, shuffle=True), make_loader("val", cfg, shuffle=False)
    pre = sm.pretrained_param_ids
    new_p = [p for p in sm.parameters() if p.requires_grad and id(p) not in pre]
    old_p = [p for p in sm.parameters() if p.requires_grad and id(p) in pre]
    groups = ([dict(params=new_p, lr=cfg.lr)] if new_p else []) + ([dict(params=old_p, lr=cfg.lr_backbone)] if old_p else [])
    if not groups:
        raise RuntimeError("nothing to train")
    opt = torch.optim.AdamW(groups, weight_decay=cfg.wd, betas=(0.9, 0.98))
    total = max(1, cfg.epochs * math.ceil(len(tr) / cfg.accum))
    warm = max(1, int(cfg.warmup * total))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else max(0.05, 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm)))))
    use_scaler = AMP_DTYPE == torch.float16
    scaler = torch.amp.GradScaler("cuda", enabled=use_scaler)
    best, bad, step = -1.0, 0, 0
    ck_path = os.path.join(local_ck, f"{tag}.pt")
    if teacher is not None:
        teacher.eval(); freeze(teacher)
    print(f"[{tag}] {len(tr)} batches/epoch x {cfg.epochs} epochs | accum {cfg.accum} | "
          f"trainable new {sum(p.numel() for p in new_p)/1e6:.2f}M + backbone {sum(p.numel() for p in old_p)/1e6:.2f}M")
    for ep in range(cfg.epochs):
        sm.train(); opt.zero_grad(set_to_none=True)
        t0, run = time.time(), 0.0
        for i, b in enumerate(tr):
            with amp_ctx():
                loss = sm.loss(b, teacher=teacher)
            if not torch.isfinite(loss):
                print(f"[{tag}] non-finite loss at batch {i}, skipped"); opt.zero_grad(set_to_none=True); continue
            scaler.scale(loss / cfg.accum).backward()
            run += float(loss)
            if (i + 1) % cfg.accum == 0 or i + 1 == len(tr):
                scaler.unscale_(opt)
                nn.utils.clip_grad_norm_([p for g in groups for p in g["params"]], cfg.clip)
                scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad(set_to_none=True); step += 1
            if (i + 1) % cfg.log_every == 0:
                print(f"[{tag}] ep {ep+1} batch {i+1}/{len(tr)} loss {run/(i+1):.4f} ({time.time()-t0:.0f}s)")
        vl = eval_loss(sm, va, max_batches=cfg.val_loss_batches)
        preds = generate_split(sm, "val", cfg, n_docs=cfg.n_eval_gen)
        m = score_split(preds, val_recs, bert=cfg.select_metric == "bertscore")
        score = m[cfg.select_metric] if m.get(cfg.select_metric) is not None else m["rougeL"]
        print(f"[{tag}] ep {ep+1:02d} train {run/len(tr):.4f} | val loss {vl:.4f} | {fmt_metrics(m)} ({time.time()-t0:.0f}s)")
        if score > best:
            best, bad = score, 0
            save_trainable(sm, ck_path, {"cfg": vars(cfg), "epoch": ep + 1, "val": m})
        else:
            bad += 1
            if bad >= cfg.patience:
                print(f"[{tag}] early stop"); break
    load_trainable(sm, ck_path)
    shutil.copy(ck_path, os.path.join(p6_dir, f"ckpt_{tag}.pt"))          # persist the best checkpoint to Drive
    return best

In [ ]:
# ============================================================
# 2. FUSION-IN-DECODER RUNNERS for the pre-trained encoder-decoder (same weights and maths as Phase 1)
#
#   doc -> K chunks (511 tokens + </s>) -> ENCODER on each chunk independently  (bidirectional, 512 window as in
#   pre-training) -> concatenate all chunk states [B, K*512, d] -> DECODER cross-attends to every chunk at once.
#
# Written as functions over the Phase-1 modules (no new weights), adding:
#   * fused SDPA attention everywhere (encoder, decoder self- and cross-attention), dropout passed to the kernel
#   * gradient checkpointing per layer, frozen bottom layers under no_grad
#   * incremental decoding: self-attention KV cache + cross-attention K/V computed ONCE per document and shared by
#     all beams (beams are folded into the query axis, so the 8k-key cross-attention cache is never copied)
#   * learned prefixes for prefix tuning (P hidden vectors per attention, projected by that layer's frozen k / v)
# ============================================================
def unwrap_attn(mod):
    return mod.inner if isinstance(mod, AdapterAfter) else mod


def rel_bias(rpb, q_len, k_len, q_offset=0):
    """T5 relative-position bias [1,H,q_len,k_len] for queries at positions q_offset.. (same bucketing as Phase 1)"""
    dev = rpb.embedding.weight.device
    q = torch.arange(q_offset, q_offset + q_len, device=dev)[:, None]
    k = torch.arange(k_len, device=dev)[None, :]
    b = RelativePositionBias._bucket(k - q, rpb.bidirectional, rpb.num_buckets, rpb.max_distance)
    return rpb.embedding(b).permute(2, 0, 1).unsqueeze(0)


def _heads(mha, x):
    B, L, _ = x.shape
    return x.view(B, L, mha.num_heads, mha.d_head).transpose(1, 2)


def attn_call(mod, xq, xkv, bias, cache=None, static=False):
    """mod: MultiHeadAttention (optionally adapter-wrapped). bias: additive [*, *, Lq, S] or None.
    static=False: self-attention; `cache`=(k,v) of previous steps is extended.
    static=True : cross-attention; `cache`=(k,v) precomputed from the encoder (may serve several beams per source).
    -> (output [B,Lq,d], (k, v))"""
    if isinstance(mod, AdapterAfter):
        out, kv = attn_call(mod.inner, xq, xkv, bias, cache, static)
        return mod.adapter(out), kv
    mha = mod
    B, Lq, D = xq.shape
    H, dh = mha.num_heads, mha.d_head
    q = _heads(mha, mha.q(xq))
    if static and cache is not None:
        k, v = cache
    else:
        k, v = _heads(mha, mha.k(xkv)), _heads(mha, mha.v(xkv))
        if cache is not None:
            k, v = torch.cat([cache[0], k], 2), torch.cat([cache[1], v], 2)
    kv = (k, v)
    pre = getattr(mha, "prefix", None)
    if pre is not None:
        P = pre.size(0)
        xp = pre.to(xq.dtype).unsqueeze(0)
        pk, pv = _heads(mha, mha.k(xp)).expand(k.size(0), -1, -1, -1), _heads(mha, mha.v(xp)).expand(k.size(0), -1, -1, -1)
        k, v = torch.cat([pk.to(k.dtype), k], 2), torch.cat([pv.to(v.dtype), v], 2)
        if bias is not None:
            bias = F.pad(bias, (P, 0), value=0.0)
    drop = mha.dropout.p if mha.training else 0.0
    Bk = k.size(0)
    if Bk != B:                                            # several beams per source: fold beams into queries
        assert Lq == 1 and B % Bk == 0
        nb = B // Bk
        qg = q.reshape(Bk, nb, H, dh).permute(0, 2, 1, 3)  # [Bk,H,nb,dh]
        o = F.scaled_dot_product_attention(qg, k, v, attn_mask=None if bias is None else bias.to(q.dtype), dropout_p=drop)
        o = o.permute(0, 2, 1, 3).reshape(B, H, 1, dh)
    else:
        o = F.scaled_dot_product_attention(q, k, v, attn_mask=None if bias is None else bias.to(q.dtype), dropout_p=drop)
    return mha.o(o.transpose(1, 2).reshape(B, Lq, H * dh)), kv


def _first_trainable(layers, x):
    lo = 0
    if torch.is_grad_enabled() and not x.requires_grad:
        while lo < len(layers) and not any(p.requires_grad for p in layers[lo].parameters()):
            lo += 1
    return lo


def ed_encode(ed, ids, am, prompt=None, ckpt=False):
    """ids [M,T] chunks, am [M,T] bool -> (states [M,T',d], mask [M,T']); T' = T + prompt length."""
    enc = ed.encoder
    x = ed.shared_embed(ids)
    if prompt is not None:
        P = prompt.size(0)
        x = torch.cat([prompt.to(x.dtype).unsqueeze(0).expand(x.size(0), -1, -1), x], 1)
        am = torch.cat([am.new_ones(am.size(0), P), am], 1)
    L = x.size(1)
    with torch.no_grad():
        dt = unwrap_attn(enc.layers[0].self_attn).q(x[:1, :1]).dtype      # autocast dtype
    bias = rel_bias(enc.pos_bias, L, L).to(dt).expand(x.size(0), -1, -1, -1).masked_fill(~am[:, None, None, :], float("-inf"))
    x = enc.dropout(x)
    lo = _first_trainable(enc.layers, x) if not bias.requires_grad else 0

    def layer_fn(x, layer):
        h = layer.ln1(x)
        a, _ = attn_call(layer.self_attn, h, h, bias)
        x = x + layer.dropout(a)
        return x + layer.dropout(layer.ff(layer.ln2(x)))

    for i, layer in enumerate(enc.layers):
        if i < lo:
            with torch.no_grad():
                x = layer_fn(x, layer)
        elif ckpt and torch.is_grad_enabled():
            x = grad_ckpt(layer_fn, x, layer, use_reentrant=False)
        else:
            x = layer_fn(x, layer)
    return enc.final_ln(x), am


def ed_decode(ed, y_in, enc_out, enc_mask, cache=None, ckpt=False):
    """y_in [B,Ly] decoder input ids; enc_out [Bs,S,d]; enc_mask [Bs,S] (Bs = B, or B/beams with a cache).
    cache: None (training) or dict(self=[(k,v)|None]*L, cross=[(k,v)]*L, len=int) -> (hidden [B,Ly,d], cache)"""
    dec = ed.decoder
    x = ed.shared_embed(y_in)
    past = 0 if cache is None else cache["len"]
    Lq = x.size(1); S_self = past + Lq
    with torch.no_grad():
        dt = unwrap_attn(dec.layers[0].self_attn).q(x[:1, :1]).dtype
    sb = rel_bias(dec.pos_bias, Lq, S_self, q_offset=past).to(dt)
    if Lq > 1:
        causal = torch.arange(S_self, device=x.device)[None, :] <= (past + torch.arange(Lq, device=x.device))[:, None]
        sb = sb.masked_fill(~causal[None, None], float("-inf"))
    cb = torch.zeros(enc_mask.shape, dtype=dt, device=x.device).masked_fill(~enc_mask, float("-inf"))[:, None, None, :]
    x = dec.dropout(x)
    new_self = []

    def layer_fn(x, layer, sc, cc):
        h = layer.ln1(x)
        a, skv = attn_call(layer.self_attn, h, h, sb, cache=sc)
        x = x + layer.dropout(a)
        h = layer.ln2(x)
        a, ckv = attn_call(layer.cross_attn, h, enc_out, cb, cache=cc, static=True)
        x = x + layer.dropout(a)
        return x + layer.dropout(layer.ff(layer.ln3(x))), skv, ckv

    for i, layer in enumerate(dec.layers):
        sc = None if cache is None else cache["self"][i]
        cc = None if cache is None else cache["cross"][i]
        if ckpt and cache is None and torch.is_grad_enabled():
            x = grad_ckpt(lambda x, layer=layer: layer_fn(x, layer, None, None)[0], x, use_reentrant=False)
        else:
            x, skv, _ = layer_fn(x, layer, sc, cc)
            new_self.append(skv)
    if cache is not None:
        cache = dict(cache, self=new_self, len=S_self)
    return dec.final_ln(x), cache


def ed_cross_cache(ed, enc_out):
    """cross-attention K/V of every decoder layer, computed once per document"""
    out = []
    for layer in ed.decoder.layers:
        mha = unwrap_attn(layer.cross_attn)
        out.append((_heads(mha, mha.k(enc_out)), _heads(mha, mha.v(enc_out))))
    return out


def reorder_dec_cache(cache, idx):
    return dict(cache, self=[None if kv is None else (kv[0].index_select(0, idx), kv[1].index_select(0, idx))
                             for kv in cache["self"]])

In [ ]:
# ============================================================
# 2b. SELF-TEST (seconds): the FiD runners reproduce the Phase-1 model exactly for a single chunk, incremental
#     decoding (self cache + shared cross cache with beams folded into queries) == full decoding.
# ============================================================
@torch.no_grad()
def check_ed_runners():
    ed = ED.to(device).eval()
    cuts = [300, 180, 300, 90]
    docs = [d for d in tr_doc if len(d) >= 300][:len(cuts)]
    B, T = len(docs), max(cuts) + 1
    ids = torch.full((B, T), PAD_ID, dtype=torch.long, device=device); am = torch.zeros(B, T, dtype=torch.bool, device=device)
    for i, (d, c) in enumerate(zip(docs, cuts)):
        ids[i, :c] = torch.as_tensor(d[:c]); ids[i, c] = EOS_ID; am[i, :c + 1] = True
    ys = [s for s in tr_sum if len(s) >= 40][:B]
    y_in = torch.tensor([[BOS_ID] + list(s[:39]) for s in ys], device=device)

    # (1) runners == LegalSeq2SeqLM.forward (original manual attention, FAST_ATTN off)
    FAST_ATTN["on"] = False
    ref = ed(ids, am.long(), y_in, torch.ones_like(y_in)).float()
    FAST_ATTN["on"] = True
    enc, em = ed_encode(ed, ids, am)
    h, _ = ed_decode(ed, y_in, enc, em)
    new = ed.lm_head(h).float()
    d1 = (ref - new).abs().max().item()
    print(f"(1) FiD runners vs Phase-1 forward      max|logit diff| {d1:.2e}")
    assert d1 < 2e-3 * max(1.0, ref.abs().max().item()), "runners do not reproduce the Phase-1 model"

    # (2) incremental decoding with 2 'beams' per source sharing one cross cache == full decoding
    nb = 2
    cache = dict(self=[None] * len(ed.decoder.layers), cross=ed_cross_cache(ed, enc), len=0)
    h0, cache = ed_decode(ed, y_in[:, :1], enc, em, cache)
    cache = reorder_dec_cache(cache, torch.arange(B, device=device).repeat_interleave(nb))
    steps = [h0[:, -1].repeat_interleave(nb, 0)]
    for t in range(1, y_in.size(1)):
        ht, cache = ed_decode(ed, y_in[:, t:t + 1].repeat_interleave(nb, 0), enc, em, cache)
        steps.append(ht[:, -1])
    inc = ed.lm_head(torch.stack(steps, 1)).float()
    d2 = (inc - new.repeat_interleave(nb, 0)).abs().max().item()
    print(f"(2) cached beam decoding vs full        max|logit diff| {d2:.2e}")
    assert d2 < 2e-3 * max(1.0, new.abs().max().item()), "incremental decoding diverges"

    # (3) FiD with 2 chunks per document: no NaN, padded chunk slots masked
    enc2 = torch.cat([enc[:2], enc[2:]], 1); em2 = torch.cat([em[:2], em[2:]], 1)
    h2, _ = ed_decode(ed, y_in[:2], enc2, em2)
    print(f"(3) FiD 2-chunk decode finite: {bool(torch.isfinite(h2).all())}")

    # (4) autocast closeness
    with amp_ctx():
        e16, m16 = ed_encode(ed, ids, am); h16, _ = ed_decode(ed, y_in, e16, m16)
    cos = F.cosine_similarity(h16.float().reshape(-1, h16.size(-1)), h.reshape(-1, h.size(-1)), -1).min().item()
    print(f"(4) autocast vs fp32 decoder states     min cosine {cos:.5f}")
    print("encoder-decoder self-test passed")


check_ed_runners()

(1) FiD runners vs Phase-1 forward      max|logit diff| 3.81e-05
(2) cached beam decoding vs full        max|logit diff| 4.58e-05
(3) FiD 2-chunk decode finite: True
(4) autocast vs fp32 decoder states     min cosine 1.00000
encoder-decoder self-test passed


In [ ]:
# ============================================================
# 3. ENCODER-DECODER SUMMARISER (FiD) + ADAPTATION METHODS
# ============================================================
ED_LINEARS = {"q", "k", "v", "o", "wi_0", "wi_1", "wo"}
N_HEADS = ED.encoder.layers[0].self_attn.num_heads
D_FF = ED.encoder.layers[0].ff.wi_0.out_features


def _is_target(name):
    return ".layers." in name and name.split(".")[-1] in ED_LINEARS


def apply_backbone_method(ed, cfg):
    meth = cfg.method
    freeze(ed)
    d = ed.shared_embed.embedding_dim
    if meth in ("full", "kd"):
        for p in ed.parameters():
            p.requires_grad = True
        return "all encoder-decoder weights"
    if meth == "partial":
        k = cfg.unfreeze_top_k
        for stack in (ed.encoder, ed.decoder):
            for layer in stack.layers[len(stack.layers) - k:]:
                for p in layer.parameters():
                    p.requires_grad = True
            for p in stack.final_ln.parameters():
                p.requires_grad = True
        return f"top {k} encoder + top {k} decoder layers + final norms"
    if meth == "lora":
        n = replace_linears(ed, lambda l: LoRALinear(l, cfg.r, cfg.alpha, cfg.lora_dropout), _is_target)
        return f"LoRA r={cfg.r} on {len(n)} linears"
    if meth == "lorma":
        n = replace_linears(ed, lambda l: LoRMALinear(l, cfg.r, cfg.alpha, cfg.lora_dropout), _is_target)
        return f"LoRMA+ r={cfg.r} on {len(n)} linears"
    if meth == "qlora":
        n = replace_linears(ed, lambda l: LoRALinear(NF4Linear(l), cfg.r, cfg.alpha, cfg.lora_dropout), _is_target)
        return f"QLoRA (NF4 base) r={cfg.r} on {len(n)} linears"
    if meth in ("houlsby", "pfeiffer"):
        for stack in (ed.encoder, ed.decoder):
            for layer in stack.layers:
                if meth == "houlsby":
                    layer.self_attn = AdapterAfter(layer.self_attn, d, cfg.adapter_dim, cfg.adapter_dropout)
                layer.ff = AdapterAfter(layer.ff, d, cfg.adapter_dim, cfg.adapter_dropout)
        return f"{meth} adapters (bottleneck {cfg.adapter_dim})"
    if meth == "prefix":
        for stack in (ed.encoder, ed.decoder):
            for layer in stack.layers:
                layer.self_attn.prefix = nn.Parameter(torch.randn(cfg.prefix_len, d) * 0.5)
        return f"prefix tuning, {cfg.prefix_len} vectors per self-attention (encoder + decoder)"
    if meth == "prompt":
        return f"prompt tuning, {cfg.prompt_len} soft tokens prepended to every encoder chunk"
    raise ValueError(f"unknown method {meth!r}")


class EDSummarizer(nn.Module):
    def __init__(self, ed, cfg):
        super().__init__()
        self.ed, self.cfg = ed, cfg
        self.pretrained_param_ids = {id(p) for p in ed.parameters()}
        self.method_desc = apply_backbone_method(ed, cfg)
        self.prompt = None
        if cfg.method == "prompt":
            init = ed.shared_embed.weight.detach()[torch.randint(1000, 20000, (cfg.prompt_len,))].clone()
            self.prompt = nn.Parameter(init)

    def encode(self, b):
        """FiD: encode the M real chunks independently, then lay them out as [B, K*T, d] per document."""
        ids, am = b["chunk_ids"].to(device), b["chunk_mask"].to(device)
        enc, em = ed_encode(self.ed, ids, am, self.prompt, ckpt=self.cfg.grad_ckpt)
        owner, slot = b["owner"].to(device), b["slot"].to(device)
        B, K, T, d = len(b["ids"]), b["k_max"], enc.size(1), enc.size(2)
        full = enc.new_zeros(B, K, T, d); fm = em.new_zeros(B, K, T)
        full[owner, slot] = enc; fm[owner, slot] = em
        return full.view(B, K * T, d), fm.view(B, K * T)

    def logits_targets(self, b):
        enc, em = self.encode(b)
        h, _ = ed_decode(self.ed, b["y_in"].to(device), enc, em, ckpt=self.cfg.grad_ckpt)
        tgt = b["tgt"].to(device)
        sel = tgt != -100
        return self.ed.lm_head(h[sel]), tgt[sel]

    def loss(self, b, teacher=None):
        logits, t = self.logits_targets(b)
        ce = F.cross_entropy(logits.float(), t, label_smoothing=self.cfg.label_smoothing)
        if teacher is None:
            return ce
        with torch.no_grad():
            tl, tt = teacher.logits_targets(b)
        assert tt.shape == t.shape
        T = self.cfg.kd_temp
        kl = F.kl_div(F.log_softmax(logits.float() / T, -1), F.log_softmax(tl.float() / T, -1),
                      log_target=True, reduction="batchmean") * (T * T)
        return (1 - self.cfg.kd_alpha) * ce + self.cfg.kd_alpha * kl

    @torch.no_grad()
    def generate(self, b, gen):
        ed = self.ed
        enc, em = self.encode(b)
        B = enc.size(0)

        def start():
            cache = dict(self=[None] * len(ed.decoder.layers), cross=ed_cross_cache(ed, enc), len=0)
            h, cache = ed_decode(ed, torch.full((B, 1), BOS_ID, dtype=torch.long, device=device), enc, em, cache)
            return cache, ed.lm_head(h[:, -1]).float()

        def step(cache, tok):
            h, cache = ed_decode(ed, tok[:, None], enc, em, cache)
            return cache, ed.lm_head(h[:, -1]).float()

        return beam_search(start, step, reorder_dec_cache, num_beams=gen["num_beams"], max_new=gen["max_new"],
                           min_new=gen["min_new"], len_penalty=gen["len_penalty"],
                           no_repeat_ngram=gen["no_repeat_ngram"], rep_penalty=gen["rep_penalty"],
                           banned=(PAD_ID, BOS_ID))


def build_student_ed(teacher_ed, n_enc, n_dec):
    """Smaller encoder-decoder initialised from evenly spaced teacher layers (shrink-and-fine-tune)."""
    d = teacher_ed.shared_embed.embedding_dim
    s = LegalSeq2SeqLM(vocab_size=teacher_ed.shared_embed.num_embeddings, d_model=d, num_heads=N_HEADS, d_ff=D_FF,
                       num_encoder_layers=n_enc, num_decoder_layers=n_dec, pad_id=PAD_ID)
    s.shared_embed.load_state_dict(teacher_ed.shared_embed.state_dict())        # lm_head is tied to it
    for src, dst, n in ((teacher_ed.encoder, s.encoder, n_enc), (teacher_ed.decoder, s.decoder, n_dec)):
        dst.pos_bias.load_state_dict(src.pos_bias.state_dict())
        dst.final_ln.load_state_dict(src.final_ln.state_dict())
        keep = np.linspace(0, len(src.layers) - 1, n).round().astype(int).tolist()
        for j, i in enumerate(keep):
            dst.layers[j].load_state_dict(src.layers[i].state_dict())
        print(f"  student {'encoder' if dst is s.encoder else 'decoder'}: teacher layers {keep}")
    print(f"student {sum(p.numel() for p in s.parameters())/1e6:.1f}M params "
          f"(teacher {sum(p.numel() for p in teacher_ed.parameters())/1e6:.1f}M)")
    return s


INPUT_KEYS = ("max_chunks", "chunk_stride", "chunk_strategy", "max_sum_tok")


def build_summarizer(cfg, backbone=None):
    ed = copy.deepcopy(ED) if backbone is None else backbone
    return EDSummarizer(ed, cfg).to(device)


def make_student_summarizer(teacher, cfg):
    return build_summarizer(cfg, backbone=build_student_ed(teacher.ed, cfg.student_enc, cfg.student_dec))


@torch.no_grad()
def check_peft_identity():
    """LoRA / LoRMA / adapters leave the model unchanged at init; QLoRA's 4-bit base stays close."""
    docs = [d for d in tr_doc if len(d) >= 200][:2]
    ids = torch.as_tensor(np.stack([d[:200] for d in docs]), dtype=torch.long, device=device)
    am = torch.ones_like(ids, dtype=torch.bool)
    y = torch.tensor([[BOS_ID] + list(s[:20]) for s in tr_sum[:2]], device=device)

    def run(ed):
        e, m = ed_encode(ed, ids, am); h, _ = ed_decode(ed, y, e, m)
        return h
    ref = run(ED.to(device).eval())
    for meth in ("lora", "lorma", "houlsby", "pfeiffer", "qlora"):
        e = copy.deepcopy(ED)
        apply_backbone_method(e, SimpleNamespace(**{**BASE_CFG, "method": meth}))
        out = run(e.to(device).eval())
        if meth == "qlora":
            c = F.cosine_similarity(out.reshape(-1, out.size(-1)), ref.reshape(-1, ref.size(-1)), -1).mean().item()
            print(f"  {meth:9s} 4-bit base: mean cosine to fp32 {c:.4f}")
        else:
            print(f"  {meth:9s} identity at init: max|diff| {(out - ref).abs().max().item():.2e}")
        del e
    torch.cuda.empty_cache()

In [ ]:
# ============================================================
# 4. DATASETS / COLLATE
#   encoder input: each chunk = 511 BPE tokens + </s> (the Phase-1 encoder format, no <s>)
#   decoder input: <s> y_1 .. y_L      target: y_1 .. y_L </s>
# ============================================================
SPLITS = {"train": (train_recs, tr_doc, tr_sum), "val": (val_recs, va_doc, va_sum), "test": (test_recs, te_doc, None)}
_DS = {}


class EDSummDS(Dataset):
    def __init__(self, split, cfg):
        recs, docs, sums = SPLITS[split]
        self.ids = [r["id"] for r in recs]
        self.chunks = [select_chunks(chunk_ids(d, cfg.chunk_len - 1, cfg.chunk_stride), cfg.max_chunks, cfg.chunk_strategy)
                       for d in docs]
        self.sums = sums

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        return self.ids[i], self.chunks[i], (None if self.sums is None else self.sums[i])


def make_collate(cfg, with_target):
    def collate(batch):
        B = len(batch)
        flat, owner, slot = [], [], []
        for i, (_, chunks, _) in enumerate(batch):
            for j, c in enumerate(chunks):
                flat.append(c); owner.append(i); slot.append(j)
        T = max(len(c) for c in flat) + 1
        ids = torch.full((len(flat), T), PAD_ID, dtype=torch.long); am = torch.zeros(len(flat), T, dtype=torch.bool)
        for m, c in enumerate(flat):
            k = len(c)
            if k:
                ids[m, :k] = torch.from_numpy(np.asarray(c, dtype=np.int64))
            ids[m, k] = EOS_ID; am[m, :k + 1] = True
        out = dict(ids=[b[0] for b in batch], chunk_ids=ids, chunk_mask=am, owner=torch.tensor(owner),
                   slot=torch.tensor(slot), k_max=max(slot) + 1)
        if with_target:
            ys = [np.asarray(b[2][:cfg.max_sum_tok], dtype=np.int64) for b in batch]
            L = max(len(y) for y in ys) + 1
            y_in = torch.full((B, L), PAD_ID, dtype=torch.long); tgt = torch.full((B, L), -100, dtype=torch.long)
            for i, s in enumerate(ys):
                n = len(s)
                y_in[i, 0] = BOS_ID; y_in[i, 1:n + 1] = torch.from_numpy(s)
                tgt[i, :n] = torch.from_numpy(s); tgt[i, n] = EOS_ID
            out.update(y_in=y_in, tgt=tgt)
        return out
    return collate


def make_loader(split, cfg, shuffle, gen=False, limit=None):
    key = (split, cfg.chunk_len, cfg.chunk_stride, cfg.max_chunks, cfg.chunk_strategy)
    if key not in _DS:
        _DS[key] = EDSummDS(split, cfg)
    ds = _DS[key]
    if limit:
        ds = torch.utils.data.Subset(ds, list(range(min(limit, len(ds)))))
    elif split == "train" and cfg.max_train_docs:
        ds = torch.utils.data.Subset(ds, list(range(min(cfg.max_train_docs, len(ds)))))
    has_y = split != "test" and not gen
    return DataLoader(ds, batch_size=cfg.gen_bs if gen else cfg.batch_size, shuffle=shuffle,
                      collate_fn=make_collate(cfg, has_y), num_workers=0, drop_last=False)

In [ ]:
# ============================================================
# 5. CONFIGURATION
# ============================================================
BASE_CFG = dict(
    method="lora",
    # FiD input: up to max_chunks windows of 511 tokens + </s>, spread over the judgment
    chunk_len=512, chunk_stride=511, max_chunks=8, chunk_strategy="uniform",
    max_sum_tok=None,             # filled in by make_cfg: min(p95 of reference lengths, MAX_DEC_LEN)
    # adaptation hyper-parameters
    r=16, alpha=32, lora_dropout=0.05, adapter_dim=64, adapter_dropout=0.1,
    prefix_len=16, prompt_len=20, unfreeze_top_k=3,
    # optimisation
    lr=5e-4, lr_backbone=5e-5, wd=0.01, epochs=5, patience=2, batch_size=2, accum=8, warmup=0.05, clip=1.0,
    label_smoothing=0.1, grad_ckpt=True, seed=42, log_every=200, max_train_docs=None,
    # model selection during training
    val_loss_batches=40, n_eval_gen=48, select_metric="rougeL", gen_bs=4, n_eval_final=None,
    # knowledge distillation (student: 6 encoder + 4 decoder layers taken from the teacher)
    teacher_ckpt=None, student_enc=6, student_dec=4, kd_alpha=0.5, kd_temp=2.0,
)
MAX_DEC_LEN = 768                 # decoder relative-position buckets are log-spaced, so lengths >512 still work

METHOD_DEFAULTS = {
    "full":     dict(lr=5e-4, lr_backbone=5e-5),
    "partial":  dict(lr=5e-4, lr_backbone=1e-4),
    "lora":     dict(lr=5e-4),
    "lorma":    dict(lr=5e-4),
    "qlora":    dict(lr=5e-4),
    "houlsby":  dict(lr=5e-4),
    "pfeiffer": dict(lr=5e-4),
    "prefix":   dict(lr=2e-3),
    "prompt":   dict(lr=3e-3),
    "kd":       dict(lr=5e-4, lr_backbone=1e-4),
}


def make_cfg(**over):
    meth = over.get("method", BASE_CFG["method"])
    c = SimpleNamespace(**{**BASE_CFG, **METHOD_DEFAULTS.get(meth, {}), **over})
    c.max_sum_tok = c.max_sum_tok or min(SUM_TOK_P95, MAX_DEC_LEN)
    return c


GEN = dict(num_beams=4, max_new=min(int(SUM_TOK_P90 * 1.1), MAX_DEC_LEN), min_new=SUM_TOK_P10, len_penalty=1.0,
           no_repeat_ngram=3, rep_penalty=1.0, max_words=None)

_c = make_cfg()
print(f"FiD input: up to {_c.max_chunks} x {_c.chunk_len} tokens | summary budget {_c.max_sum_tok} tokens "
      f"(reference p95 {SUM_TOK_P95}) | generation {GEN['min_new']}-{GEN['max_new']} tokens, {GEN['num_beams']} beams")
_cov = np.mean(np.minimum(dl, _c.max_chunks * (_c.chunk_len - 1)) / np.maximum(dl, 1))
print(f"fraction of document tokens the encoder sees (mean over train): {_cov:.0%}")

check_peft_identity()

FiD input: up to 8 x 512 tokens | summary budget 768 tokens (reference p95 2390) | generation 340-768 tokens, 4 beams
fraction of document tokens the encoder sees (mean over train): 84%
  lora      identity at init: max|diff| 0.00e+00
  lorma     identity at init: max|diff| 0.00e+00
  houlsby   identity at init: max|diff| 0.00e+00
  pfeiffer  identity at init: max|diff| 0.00e+00
  qlora     4-bit base: mean cosine to fp32 0.9997


In [ ]:
# ============================================================
# 6. EXPERIMENT RUNNER
# ============================================================
RESULTS = {}
_res_path = os.path.join(p6_dir, "results.json")
if os.path.exists(_res_path):
    RESULTS = json.load(open(_res_path))


def _teacher_from_ckpt(path):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    tcfg = SimpleNamespace(**ck["cfg"])
    t = build_summarizer(tcfg)
    load_trainable(t, path)
    return t, tcfg


def evaluate_and_submit(sm, cfg, name, submit=True):
    t0 = time.time()
    val_pred = generate_split(sm, "val", cfg, n_docs=getattr(cfg, "n_eval_final", None))
    m = score_split(val_pred, val_recs, bert=True)
    print(f"[{name}] VAL ({len(val_pred)} docs): {fmt_metrics(m)} ({time.time()-t0:.0f}s)")
    RESULTS[name] = dict(val=m, method=cfg.method, desc=sm.method_desc,
                         trainable=n_params(sm), total=n_params(sm, trainable=False),
                         cfg={k: v for k, v in vars(cfg).items() if isinstance(v, (int, float, str, bool, type(None)))})
    json.dump(RESULTS, open(_res_path, "w"), indent=1)
    json.dump(val_pred, open(os.path.join(p6_dir, f"valpred_{name}.json"), "w"), ensure_ascii=False)
    if submit:
        test_pred = generate_split(sm, "test", cfg)
        write_submission(test_pred, name)
    return m


def run_experiment(name, submit=True, **over):
    cfg = make_cfg(**over)
    set_seed(cfg.seed)
    teacher = None
    if cfg.method == "kd":
        assert cfg.teacher_ckpt and os.path.exists(cfg.teacher_ckpt), "kd needs teacher_ckpt (e.g. the full fine-tune)"
        teacher, tcfg = _teacher_from_ckpt(cfg.teacher_ckpt)
        for k in INPUT_KEYS:
            setattr(cfg, k, getattr(tcfg, k))                       # student reads exactly the teacher's inputs
        sm = make_student_summarizer(teacher, cfg)
    else:
        sm = build_summarizer(cfg)
    print(f"\n=== {name} | {sm.method_desc} | trainable {n_params(sm)/1e6:.2f}M / {n_params(sm, False)/1e6:.1f}M ===")
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    train_summarizer(sm, cfg, name, teacher=teacher)
    train_min = (time.time() - t0) / 60
    peak = torch.cuda.max_memory_allocated() / 2**30 if device.type == "cuda" else 0
    del teacher; gc.collect(); torch.cuda.empty_cache()
    m = evaluate_and_submit(sm, cfg, name, submit)
    RESULTS[name].update(train_min=train_min, peak_gb=peak)
    json.dump(RESULTS, open(_res_path, "w"), indent=1)
    del sm; gc.collect(); torch.cuda.empty_cache()
    return m


def resubmit(name, **gen_over):
    """Re-generate val + test from a saved checkpoint with different decoding (no retraining)."""
    path = os.path.join(p6_dir, f"ckpt_{name}.pt")
    ck = torch.load(path, map_location="cpu", weights_only=False)
    cfg = SimpleNamespace(**ck["cfg"])
    if cfg.method == "kd":
        teacher, _ = _teacher_from_ckpt(cfg.teacher_ckpt)
        sm = make_student_summarizer(teacher, cfg); del teacher
    else:
        sm = build_summarizer(cfg)
    load_trainable(sm, path)
    saved = dict(GEN); GEN.update(gen_over)
    tag = name + "_" + "_".join(f"{k}{v}" for k, v in gen_over.items())
    try:
        return evaluate_and_submit(sm, cfg, tag)
    finally:
        GEN.clear(); GEN.update(saved); del sm; gc.collect(); torch.cuda.empty_cache()


def show_results():
    print(f"\n{'experiment':26s} {'method':9s} {'BERTScore':>9s} {'R1':>7s} {'R2':>7s} {'RL':>7s} {'words':>6s} "
          f"{'train M':>8s} {'total M':>8s} {'min':>6s} {'GB':>5s}")
    for n, r in RESULTS.items():
        v = r["val"]; bs = v.get("bertscore")
        print(f"{n:26s} {r['method']:9s} {('%.4f' % bs) if bs is not None else 'n/a':>9s} {v['rouge1']:7.4f} "
              f"{v['rouge2']:7.4f} {v['rougeL']:7.4f} {v['avg_words']:6.0f} {r['trainable']/1e6:8.2f} "
              f"{r['total']/1e6:8.1f} {r.get('train_min', 0):6.0f} {r.get('peak_gb', 0):5.1f}")

In [ ]:
# ============================================================
# 7. RUN
# ============================================================
# (0) Smoke test FIRST (a few minutes): whole pipeline on 64 documents, no submission.
run_experiment("E0_smoke", method="lora", max_train_docs=64, epochs=1, n_eval_gen=8, n_eval_final=16, submit=False)

# (1) Six substantially different adaptation methods (LoRMA compulsory, KD mandatory). Same FiD input and
#     decoding, so the differences come from the adaptation method.
# run_experiment("E1_full",    method="full")                              # strongest; also the KD teacher
# run_experiment("E2_lora",    method="lora")
# run_experiment("E3_lorma",   method="lorma")                             # compulsory
# run_experiment("E4_houlsby", method="houlsby")
# run_experiment("E5_prefix",  method="prefix")
# run_experiment("E6_kd",      method="kd", teacher_ckpt=os.path.join(p6_dir, "ckpt_E1_full.pt"),
#                student_enc=6, student_dec=4)

# (2) Optional extras / ablations
# run_experiment("E7_qlora",    method="qlora")                            # 4-bit base: memory vs quality
# run_experiment("E8_pfeiffer", method="pfeiffer")
# run_experiment("E9_prompt",   method="prompt")
# run_experiment("E10_partial", method="partial", unfreeze_top_k=3)
# run_experiment("E11_k4",      method="lora", max_chunks=4)               # how much of the document is needed
# run_experiment("E12_k16",     method="lora", max_chunks=16, batch_size=1, accum=16)
# run_experiment("E13_head",    method="lora", chunk_strategy="head_tail")

# (3) Decoding sweeps on a trained checkpoint (no retraining):
# resubmit("E3_lorma", len_penalty=1.3)
# resubmit("E3_lorma", num_beams=1)
# resubmit("E3_lorma", min_new=300)

show_results()


=== E0_smoke | LoRA r=16 on 162 linears | trainable 3.98M / 105.7M ===
[E0_smoke] 32 batches/epoch x 1 epochs | accum 8 | trainable new 3.98M + backbone 0.00M


/tmp/ipykernel_744/1471079892.py:100: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  run += float(loss)


[E0_smoke] ep 01 train 7.3073 | val loss 6.4044 | BERTScore n/a | R1 0.3967 R2 0.1344 RL 0.2077 | words 661 (ref 501) (80s)


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

... (3 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[E0_smoke] VAL (16 docs): BERTScore 0.7908 | R1 0.4034 R2 0.1348 RL 0.2074 | words 656 (ref 521) (125s)

experiment                 method    BERTScore      R1      R2      RL  words  train M  total M    min    GB
E0_smoke                   lora         0.7908  0.4034  0.1348  0.2074    656     3.98    105.7      1   1.8


In [ ]:
run_experiment("E1_full",    method="full")                              # strongest; also the KD teacher
run_experiment("E2_lora",    method="lora")
run_experiment("E3_lorma",   method="lorma")                             # compulsory
run_experiment("E4_houlsby", method="houlsby")
run_experiment("E5_prefix",  method="prefix")
run_experiment("E6_kd",      method="kd", teacher_ckpt=os.path.join(p6_dir, "ckpt_E1_full.pt"),
               student_enc=6, student_dec=4)


show_results()


=== E1_full | all encoder-decoder weights | trainable 101.74M / 101.7M ===
[E1_full] 3339 batches/epoch x 5 epochs | accum 8 | trainable new 0.00M + backbone 101.74M
[E1_full] ep 1 batch 200/3339 loss 7.1441 (120s)
[E1_full] ep 1 batch 400/3339 loss 6.7871 (239s)
[E1_full] ep 1 batch 600/3339 loss 6.6257 (356s)
[E1_full] ep 1 batch 800/3339 loss 6.5218 (475s)
[E1_full] ep 1 batch 1000/3339 loss 6.4401 (589s)
[E1_full] ep 1 batch 1200/3339 loss 6.3865 (708s)
[E1_full] ep 1 batch 1400/3339 loss 6.3408 (826s)
[E1_full] ep 1 batch 1600/3339 loss 6.3017 (945s)
[E1_full] ep 1 batch 1800/3339 loss 6.2672 (1065s)
[E1_full] ep 1 batch 2000/3339 loss 6.2342 (1182s)
[E1_full] ep 1 batch 2200/3339 loss 6.2035 (1301s)
[E1_full] ep 1 batch 2400/3339 loss 6.1765 (1420s)
[E1_full] ep 1 batch 2600/3339 loss 6.1505 (1538s)
[E1_full] ep 1 batch 2800/3339 loss 6.1303 (1657s)
[E1_full] ep 1 batch 3000/3339 loss 6.1090 (1775s)
[E1_full] ep 1 batch 3200/3339 loss 6.0924 (1896s)
    generated 40 (141s)
[E1_f

/tmp/ipykernel_744/1471079892.py:104: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad(set_to_none=True); step += 1


[E3_lorma] ep 1 batch 200/3339 loss 7.0835 (134s)
[E3_lorma] ep 1 batch 400/3339 loss 6.7644 (268s)
[E3_lorma] ep 1 batch 600/3339 loss 6.6255 (402s)
[E3_lorma] ep 1 batch 800/3339 loss 6.5489 (536s)
[E3_lorma] ep 1 batch 1000/3339 loss 6.5123 (670s)
[E3_lorma] ep 1 batch 1200/3339 loss 6.4784 (806s)
[E3_lorma] ep 1 batch 1400/3339 loss 6.4531 (937s)
[E3_lorma] ep 1 batch 1600/3339 loss 6.4262 (1072s)
[E3_lorma] ep 1 batch 1800/3339 loss 6.4050 (1203s)
[E3_lorma] ep 1 batch 2000/3339 loss 6.3870 (1334s)
[E3_lorma] ep 1 batch 2200/3339 loss 6.3712 (1467s)
[E3_lorma] ep 1 batch 2400/3339 loss 6.3613 (1598s)
[E3_lorma] ep 1 batch 2600/3339 loss 6.3491 (1731s)
[E3_lorma] ep 1 batch 2800/3339 loss 6.3390 (1864s)
[E3_lorma] ep 1 batch 3000/3339 loss 6.3301 (1998s)
[E3_lorma] ep 1 batch 3200/3339 loss 6.3219 (2130s)
    generated 40 (259s)
[E3_lorma] ep 01 train 6.3185 | val loss 5.9403 | BERTScore n/a | R1 0.4502 R2 0.1727 RL 0.2196 | words 464 (ref 677) (2543s)
[E3_lorma] ep 2 batch 200/333

In [ ]:
run_experiment("E4_houlsby", method="houlsby")
run_experiment("E5_prefix",  method="prefix")
run_experiment("E6_kd",      method="kd", teacher_ckpt=os.path.join(p6_dir, "ckpt_E1_full.pt"),
               student_enc=6, student_dec=4)


show_results()


=== E4_houlsby | houlsby adapters (bottleneck 64) | trainable 2.38M / 104.1M ===
[E4_houlsby] 3339 batches/epoch x 5 epochs | accum 8 | trainable new 2.38M + backbone 0.00M


/tmp/ipykernel_917/1471079892.py:100: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  run += float(loss)


[E4_houlsby] ep 1 batch 200/3339 loss 8.0964 (107s)
[E4_houlsby] ep 1 batch 400/3339 loss 7.3429 (216s)
[E4_houlsby] ep 1 batch 600/3339 loss 7.0474 (323s)
[E4_houlsby] ep 1 batch 800/3339 loss 6.8916 (430s)
[E4_houlsby] ep 1 batch 1000/3339 loss 6.7906 (535s)
[E4_houlsby] ep 1 batch 1200/3339 loss 6.7179 (644s)
[E4_houlsby] ep 1 batch 1400/3339 loss 6.6675 (749s)
[E4_houlsby] ep 1 batch 1600/3339 loss 6.6231 (856s)
[E4_houlsby] ep 1 batch 1800/3339 loss 6.5907 (965s)
[E4_houlsby] ep 1 batch 2000/3339 loss 6.5605 (1072s)
[E4_houlsby] ep 1 batch 2200/3339 loss 6.5372 (1180s)
[E4_houlsby] ep 1 batch 2400/3339 loss 6.5159 (1287s)
[E4_houlsby] ep 1 batch 2600/3339 loss 6.4981 (1394s)
[E4_houlsby] ep 1 batch 2800/3339 loss 6.4834 (1502s)
[E4_houlsby] ep 1 batch 3000/3339 loss 6.4699 (1608s)
[E4_houlsby] ep 1 batch 3200/3339 loss 6.4588 (1715s)
    generated 40 (173s)
[E4_houlsby] ep 01 train 6.4516 | val loss 6.0126 | BERTScore n/a | R1 0.4367 R2 0.1674 RL 0.2186 | words 484 (ref 677) (2005

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

... (3 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[E4_houlsby] VAL (351 docs): BERTScore 0.8068 | R1 0.4229 R2 0.1658 RL 0.2118 | words 467 (ref 825) (1589s)
    generated 40 (171s)
    generated 80 (340s)
wrote 100 summaries (avg 447 words) -> /content/drive/MyDrive/Phase1_Pretraining/phase1_encoder_decoder/phase6/submission_E4_houlsby/submission.zip

=== E5_prefix | prefix tuning, 16 vectors per self-attention (encoder + decoder) | trainable 0.15M / 101.9M ===
[E5_prefix] 3339 batches/epoch x 5 epochs | accum 8 | trainable new 0.15M + backbone 0.00M
[E5_prefix] ep 1 batch 200/3339 loss 8.9746 (114s)
[E5_prefix] ep 1 batch 400/3339 loss 8.8471 (227s)
[E5_prefix] ep 1 batch 600/3339 loss 8.6437 (341s)
[E5_prefix] ep 1 batch 800/3339 loss 8.3715 (455s)
[E5_prefix] ep 1 batch 1000/3339 loss 8.0926 (567s)
[E5_prefix] ep 1 batch 1200/3339 loss 7.8747 (680s)
[E5_prefix] ep 1 batch 1400/3339 loss 7.7087 (794s)
[E5_prefix] ep 1 batch 1600/3339 loss 7.5808 (907s)
[E5_prefix] ep 1 batch 1800/3339 loss 7.4803 (1021s)
[E5_prefix] ep 1 batch 2000